In [1]:
!git clone https://github.com/nvdat16/MultiModal-Sentiment.git

Cloning into 'MultiModal-Sentiment'...
remote: Enumerating objects: 96, done.
remote: Counting objects: 100% (96/96), done.
remote: Compressing objects: 100% (67/67), done.
remote: Total 96 (delta 30), reused 82 (delta 16), pack-reused 0 (from 0)
Receiving objects: 100% (96/96), 2.91 MiB | 30.38 MiB/s, done.
Resolving deltas: 100% (30/30), done.


In [2]:
%cd /content/MultiModal-Sentiment

/content/MultiModal-Sentiment


In [3]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("dunyajasim/twitter-dataset-for-sentiment-analysis")

print("Path to dataset files:", path)

100%|██████████| 201M/201M [00:01<00:00, 141MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/dunyajasim/twitter-dataset-for-sentiment-analysis/versions/1


In [4]:
import os

# Đường dẫn nguồn và đích
source_path = "/root/.cache/kagglehub/datasets/dunyajasim/twitter-dataset-for-sentiment-analysis/versions/1"
destination_path = "/content/MultiModal-Sentiment/data"

# Tạo thư mục data nếu chưa có
!mkdir -p {destination_path}

# Di chuyển toàn bộ nội dung vào project
!mv {source_path}/* {destination_path}

print(f"Đã chuyển dữ liệu vào: {destination_path}")

Đã chuyển dữ liệu vào: /content/MultiModal-Sentiment/data


In [21]:
!python src/dataset/build_data.py

Total samples: 4869
label
neutral     1771
positive    1646
negative    1452
Name: count, dtype: int64
Load dataset: Done


In [28]:
!python -m src.tools.train --num_epoch 5 --batch_size 32 --mode 'multimodal'

Total samples: 4869
label
neutral     1771
positive    1646
negative    1452
Name: count, dtype: int64
Loading weights: 100% 199/199 [00:00<00:00, 1375.59it/s, Materializing param=pooler.dense.weight]
BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: 

In [29]:
!python -m src.tools.train --num_epoch 5 --batch_size 32 --mode 'image'

Total samples: 4869
label
neutral     1771
positive    1646
negative    1452
Name: count, dtype: int64
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Epoch 1/5: 100% 122/122 [00:23<00:00,  5.27it/s]
Train Loss: 0.5453
Validating: 100% 31/31 [00:06<00:00,  5.03it/s]

Validation Results
------------------------------
Validation Loss: 0.5383
Accuracy: 0.4035
F1-Score: 0.3890
Classification Report
              precision    

In [26]:
!python -m src.tools.train --num_epoch 5 --batch_size 32 --mode 'text'

Total samples: 4869
label
neutral     1771
positive    1646
negative    1452
Name: count, dtype: int64
Loading weights: 100% 199/199 [00:00<00:00, 1002.99it/s, Materializing param=pooler.dense.weight]
BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Epoch 1/5: 100% 122/122 [00:25<00:00,  4.81it/s]
Train Loss: 0.4401
Valida

In [ ]:
!pip install fastapi uvicorn pyngrok python-multipart nest_asyncio torch pillow

In [ ]:
!ngrok config add-authtoken "Your_Ngrok_Auth_Token"

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [ ]:
import torch
from PIL import Image
from torchvision import transforms

# device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

from src.model import build_model

model = build_model(mode="multimodal", n_classes=3)
model.load_state_dict(torch.load("/content/best_model_multimodal.pth", map_location=device))
model.to(device)
model.eval()

text_model = build_model(mode="text", n_classes=3)
text_model.load_state_dict(torch.load("/content/best_model_text.pth", map_location=device))
text_model.to(device)
text_model.eval()

image_model = build_model(mode="image", n_classes=3)
image_model.load_state_dict(torch.load("/content/best_model_image.pth", map_location=device))
image_model.to(device)
image_model.eval()

# transform ảnh
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],std=[0.229, 0.224, 0.225]),
])

from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/mod

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
from fastapi import FastAPI, UploadFile, File, Form
from PIL import Image
import io

app = FastAPI()

@app.get("/")
def home():
    return {"message": "API is running"}

@app.post("/predict")
async def predict(
    text: str = Form(None),
    file: UploadFile = File(None)
):
    try:
        # 1. Tiền xử lý dữ liệu
        input_ids, attention_mask = None, None
        image_tensor = None
        label_map = {0: "negative", 1: "neutral", 2: "positive"}

        # Xử lý Text
        if text:
            inputs = tokenizer(text, return_tensors="pt", padding="max_length",
                               truncation=True, max_length=32).to(device)
            input_ids = inputs["input_ids"]
            attention_mask = inputs["attention_mask"]

        # Xử lý Image
        if file:
            image_bytes = await file.read()
            image = Image.open(io.BytesIO(image_bytes)).convert("RGB")
            image_tensor = transform(image).unsqueeze(0).to(device)

        # 2. Dự đoán
        with torch.no_grad():
            # --- Nhánh Multimodal ---
            logits_multi = model(input_ids=input_ids, attention_mask=attention_mask, images=image_tensor)
            probs_multi = torch.softmax(logits_multi, dim=1)

            # --- Nhánh Text-only ---
            logits_text = text_model(input_ids=input_ids, attention_mask=attention_mask)
            probs_text = torch.softmax(logits_text, dim=1)

            # --- Nhánh Image-only ---
            logits_img = image_model(image_tensor)
            probs_img = torch.softmax(logits_img, dim=1)

        # 3. Trích xuất kết quả
        # Multimodal
        pred_multi = torch.argmax(probs_multi, dim=1).item()
        conf_multi = probs_multi.max().item()
        dist_multi = probs_multi[0].tolist() # [neg, neu, pos]

        # Text only
        pred_t = torch.argmax(probs_text, dim=1).item()
        conf_t = probs_text.max().item()

        # Image only
        pred_i = torch.argmax(probs_img, dim=1).item()
        conf_i = probs_img.max().item()

        return {
            "sentiment": label_map[pred_multi],
            "confidence": round(conf_multi * 100, 2),
            "negative": round(dist_multi[0] * 100, 2),
            "neutral": round(dist_multi[1] * 100, 2),
            "positive": round(dist_multi[2] * 100, 2),

            # Kết quả thực tế từ checkpoint riêng
            "text_only_sentiment": label_map[pred_t],
            "text_only_confidence": round(conf_t * 100, 2),
            "image_only_sentiment": label_map[pred_i],
            "image_only_confidence": round(conf_i * 100, 2),

            "reasoning": f"Kết quả Multimodal đạt độ tin cậy {round(conf_multi*100,1)}%, "
                         f"trong khi Text đơn lẻ đạt {round(conf_t*100,1)}%."
        }

    except Exception as e:
        return {"error": str(e)}

In [ ]:
from pyngrok import ngrok
import nest_asyncio

nest_asyncio.apply()

public_url = ngrok.connect(8000)
print("Public URL:", public_url)

Public URL: NgrokTunnel: "https://hoa-unlicentiated-pablo.ngrok-free.dev" -> "http://localhost:8000"


In [ ]:
import uvicorn
import asyncio

config = uvicorn.Config(app, host="0.0.0.0", port=8000)
server = uvicorn.Server(config)

asyncio.run(server.serve())

INFO:     Started server process [5263]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     2402:800:6d73:4d4:b0e6:6873:dfc7:765:0 - "POST /predict HTTP/1.1" 200 OK
INFO:     2402:800:6d73:4d4:b0e6:6873:dfc7:765:0 - "POST /predict HTTP/1.1" 200 OK
INFO:     2402:800:6d73:4d4:b0e6:6873:dfc7:765:0 - "POST /predict HTTP/1.1" 200 OK
INFO:     2402:800:6d73:4d4:b0e6:6873:dfc7:765:0 - "POST /predict HTTP/1.1" 200 OK
INFO:     2402:800:6d73:4d4:b0e6:6873:dfc7:765:0 - "POST /predict HTTP/1.1" 200 OK
INFO:     2402:800:6d73:4d4:b0e6:6873:dfc7:765:0 - "POST /predict HTTP/1.1" 200 OK
INFO:     2402:800:6d73:4d4:b0e6:6873:dfc7:765:0 - "POST /predict HTTP/1.1" 200 OK
INFO:     2402:800:6d73:4d4:b0e6:6873:dfc7:765:0 - "POST /predict HTTP/1.1" 200 OK
INFO:     2402:800:6d73:4d4:b0e6:6873:dfc7:765:0 - "POST /predict HTTP/1.1" 200 OK
INFO:     2402:800:6d73:4d4:b0e6:6873:dfc7:765:0 - "POST /predict HTTP/1.1" 200 OK
